# 🧬 NonBScanner - Local Analysis Workflow (Unlimited)

**Complete workflow for Non-B DNA motif detection and analysis**

## 🚀 Local Version Advantages

| Feature | Local Version | Web Version |
|---------|--------------|-------------|
| **Sequence Length** | ✅ **UNLIMITED** | 1,000,000 nt max |
| **Genome-scale Analysis** | ✅ Full genomes | Chunked analysis |
| **Processing** | Multi-core parallel | Single-threaded |
| **Memory** | Local RAM | Cloud limited |

This notebook provides a streamlined approach to:
1. **Setup & Analysis**: Import libraries and analyze FASTA sequences of ANY size
2. **Results & Export**: View results and export to Excel/CSV
3. **Visualization**: Generate comprehensive plots and statistics

---

## Motif Detection Parameters

| Motif Type | Arm/Unit Length | Spacer/Loop | Additional Requirements |
|------------|-----------------|-------------|-------------------------|
| **Cruciform DNA** | 10–100 nt | 0–3 nt | Reverse complement arms |
| **Slipped DNA** | 10–50 nt repeat | 0 nt (adjacent) | Direct repeat units |
| **Triplex DNA** | 10–100 nt mirrored | 0–8 nt | ≥90% Purine or Pyrimidine |

---

## Box 1: Setup and Analysis

### Import Libraries and Analyze Sequences

In [ ]:
# ============================================================================
# BOX 1: SETUP AND ANALYSIS (UNLIMITED SEQUENCE LENGTH)
# ============================================================================
# NOTE: This local version has NO sequence length limits!
# It can process genome-scale sequences (100MB+) on your local hardware.
# ============================================================================

import time
start_time = time.time()

# Import NonBScanner modules
from nonbscanner import analyze_sequence, analyze_multiple_sequences, get_motif_info
from utilities import (
    read_fasta_file, export_to_excel, export_to_csv, export_to_json,
    export_results_to_dataframe, get_basic_stats, calculate_motif_statistics
)
from visualizations import (
    plot_motif_distribution, plot_coverage_map, plot_nested_pie_chart,
    plot_length_distribution, plot_score_distribution,
    plot_circos_motif_density, plot_radial_class_density, plot_stacked_density_track
)
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

print("✅ Libraries imported successfully!")
print("\n🚀 LOCAL VERSION: UNLIMITED sequence length support")
print("   (Web version limited to 1,000,000 nucleotides)\n")
print("📋 Available Motif Classes:")
motif_info = get_motif_info()
print(f"Total Classes: {motif_info.get('total_classes', 0)}")
classification = motif_info.get('classification', {})
for cls_id, info in list(classification.items())[:5]:
    print(f"  - {info['name']}: {', '.join(info['subclasses'][:2])}...")
print(f"  ... and {len(classification) - 5} more!\n")

# ============================================================================
# ANALYZE YOUR FASTA FILE (NO SIZE LIMIT)
# ============================================================================

# Specify your FASTA file path
fasta_file = "example_motifs_multiline.fasta"  # Change this to your file

# Parse FASTA file
sequences = read_fasta_file(fasta_file)
print(f"📄 Loaded {len(sequences)} sequence(s) from {fasta_file}\n")

# Analyze all sequences with performance tracking
all_results = []
analysis_times = []
for seq_name, seq in sequences.items():
    seq_start = time.time()
    print(f"🔬 Analyzing: {seq_name} ({len(seq):,} bp)...")
    results = analyze_sequence(seq, seq_name)
    seq_time = time.time() - seq_start
    analysis_times.append({'name': seq_name, 'length': len(seq), 'time': seq_time, 'motifs': len(results)})
    all_results.append({
        'name': seq_name,
        'sequence': seq,
        'motifs': results
    })
    speed = len(seq) / seq_time if seq_time > 0 else 0
    print(f"   ✓ Found {len(results)} motifs in {seq_time:.2f}s ({speed:,.0f} bp/s)\n")

# Combine all motifs
all_motifs = []
total_sequence_length = 0
for result in all_results:
    total_sequence_length += len(result['sequence'])
    for motif in result['motifs']:
        if 'Sequence_Name' not in motif:
            motif['Sequence_Name'] = result['name']
        all_motifs.append(motif)

total_time = time.time() - start_time
overall_speed = total_sequence_length / total_time if total_time > 0 else 0

print("\n" + "=" * 80)
print("✅ ANALYSIS COMPLETE - PERFORMANCE SUMMARY")
print("=" * 80)
print(f"📊 Total motifs detected: {len(all_motifs)}")
print(f"🧬 Sequences analyzed: {len(all_results)}")
print(f"📏 Total sequence length: {total_sequence_length:,} bp")
print(f"⏱️  Total processing time: {total_time:.2f} seconds")
print(f"🚀 Overall speed: {overall_speed:,.0f} bp/second")
print(f"📈 Motif density: {len(all_motifs) / (total_sequence_length / 1000):.2f} motifs/kb")

---

## Box 2: Results and Export

### View Results, Summary Statistics, and Export Data

In [ ]:
# ============================================================================
# BOX 2: RESULTS AND EXPORT (ENHANCED PERFORMANCE REPORTING)
# ============================================================================

# Convert results to DataFrame for viewing
df_results = export_results_to_dataframe(all_motifs)

print("=" * 80)
print("📊 COMPREHENSIVE RESULTS SUMMARY")
print("=" * 80)

# Enhanced performance metrics display
sequence_kb = total_sequence_length / 1000
print("\n⚡ PERFORMANCE METRICS:")
print(f"  ├─ Total Processing Time: {total_time:.2f} seconds")
print(f"  ├─ Processing Speed: {overall_speed:,.0f} bp/second")
print(f"  ├─ Sequences Analyzed: {len(all_results)}")
print(f"  └─ Total Base Pairs: {total_sequence_length:,} bp ({sequence_kb:.2f} kb)")

# Display summary by class with density (motifs/kb)
class_counts = df_results['Class'].value_counts()
print("\n📈 MOTIF DISTRIBUTION BY CLASS:")
print(f"  {'Class':<30} {'Count':>8} {'Density':>12} {'% of Total':>12}")
print(f"  {'-'*30} {'-'*8} {'-'*12} {'-'*12}")
for cls, count in class_counts.items():
    density = count / sequence_kb if sequence_kb > 0 else 0
    percentage = (count / len(all_motifs) * 100) if len(all_motifs) > 0 else 0
    print(f"  {cls:<30} {count:>8} {density:>10.2f}/kb {percentage:>10.1f}%")

# Display summary statistics
print("\n📏 SUMMARY STATISTICS:")
print(f"  ├─ Total Motifs: {len(all_motifs)}")
print(f"  ├─ Overall Density: {len(all_motifs) / sequence_kb:.2f} motifs/kb")
print(f"  ├─ Unique Classes: {df_results['Class'].nunique()}")
print(f"  ├─ Unique Subclasses: {df_results['Subclass'].nunique()}")
print(f"  ├─ Average Length: {df_results['Length'].mean():.1f} bp")
print(f"  ├─ Average Score: {df_results['Score'].mean():.3f}")
print(f"  ├─ Score Range: {df_results['Score'].min():.3f} - {df_results['Score'].max():.3f}")
print(f"  └─ Length Range: {df_results['Length'].min()} - {df_results['Length'].max()} bp")

# Per-sequence analysis summary
print("\n🧬 PER-SEQUENCE ANALYSIS:")
print(f"  {'Sequence':<30} {'Length':>12} {'Motifs':>8} {'Density':>12} {'Time':>10}")
print(f"  {'-'*30} {'-'*12} {'-'*8} {'-'*12} {'-'*10}")
for at in analysis_times:
    seq_density = at['motifs'] / (at['length'] / 1000) if at['length'] > 0 else 0
    print(f"  {at['name'][:30]:<30} {at['length']:>10,} bp {at['motifs']:>8} {seq_density:>10.2f}/kb {at['time']:>8.2f}s")

# Display first few rows
print("\n📋 First 10 Motifs (Preview):")
display(df_results[['Sequence_Name', 'Class', 'Subclass', 'Start', 'End', 'Length', 'Score']].head(10))

# ============================================================================
# EXPORT RESULTS
# ============================================================================

print("\n" + "=" * 80)
print("💾 EXPORTING RESULTS")
print("=" * 80)

# Export to Excel (multi-sheet with class breakdown)
excel_file = "nbdscanner_results.xlsx"
export_to_excel(all_motifs, excel_file)
print(f"✅ Excel file saved: {excel_file}")
print("   (Contains multiple sheets: Consolidated + per-class breakdown)")

# Export to CSV
csv_file = "nbdscanner_results.csv"
csv_data = export_to_csv(all_motifs)
with open(csv_file, 'w') as f:
    f.write(csv_data)
print(f"✅ CSV file saved: {csv_file}")

# Export to JSON
json_file = "nbdscanner_results.json"
json_data = export_to_json(all_motifs, pretty=True)
with open(json_file, 'w') as f:
    f.write(json_data)
print(f"✅ JSON file saved: {json_file}")

print("\n🎉 All exports complete!")

---

## Box 3: Comprehensive Visualization

### Generate Publication-Quality Plots and Analysis

In [ ]:
# ============================================================================
# BOX 3: VISUALIZATION AND ANALYSIS
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

# Use a compatible style that works across matplotlib versions
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    try:
        plt.style.use('seaborn-darkgrid')
    except:
        sns.set_style('darkgrid')

print("📊 GENERATING VISUALIZATIONS")
print("=" * 80)

# Set up figure layout
fig = plt.figure(figsize=(20, 12))

# ============================================================================
# 1. MOTIF CLASS DISTRIBUTION
# ============================================================================
print("\n1️⃣ Creating Motif Class Distribution...")
ax1 = plt.subplot(2, 3, 1)

class_counts = Counter([m.get('Class', 'Unknown') for m in all_motifs])
classes = list(class_counts.keys())
counts = list(class_counts.values())
ax1.bar(range(len(classes)), counts, color='steelblue', alpha=0.7)
ax1.set_xticks(range(len(classes)))
ax1.set_xticklabels(classes, rotation=45, ha='right')
ax1.set_xlabel('Motif Class')
ax1.set_ylabel('Count')
ax1.set_title('Motif Class Distribution', fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# ============================================================================
# 2. SUBCLASS DISTRIBUTION (TOP 10)
# ============================================================================
print("2️⃣ Creating Subclass Distribution...")
ax2 = plt.subplot(2, 3, 2)
subclass_counts = Counter([m.get('Subclass', 'Unknown') for m in all_motifs])
top_subclasses = dict(subclass_counts.most_common(10))
ax2.barh(list(top_subclasses.keys()), list(top_subclasses.values()), 
         color='coral', alpha=0.7)
ax2.set_xlabel('Count')
ax2.set_title('Top 10 Subclasses', fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

# ============================================================================
# 3. LENGTH DISTRIBUTION
# ============================================================================
print("3️⃣ Creating Length Distribution...")
ax3 = plt.subplot(2, 3, 3)
lengths = [m.get('Length', 0) for m in all_motifs]
ax3.hist(lengths, bins=30, color='green', alpha=0.7, edgecolor='black')
ax3.set_xlabel('Length (bp)')
ax3.set_ylabel('Frequency')
ax3.set_title('Motif Length Distribution', fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

# ============================================================================
# 4. SCORE DISTRIBUTION
# ============================================================================
print("4️⃣ Creating Score Distribution...")
ax4 = plt.subplot(2, 3, 4)
scores = [m.get('Score', 0) for m in all_motifs]
ax4.hist(scores, bins=30, color='purple', alpha=0.7, edgecolor='black')
ax4.set_xlabel('Score')
ax4.set_ylabel('Frequency')
ax4.set_title('Motif Score Distribution', fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

# ============================================================================
# 5. CLASS PIE CHART
# ============================================================================
print("5️⃣ Creating Class Pie Chart...")
ax5 = plt.subplot(2, 3, 5)
ax5.pie(counts, labels=classes, autopct='%1.1f%%', startangle=90)
ax5.set_title('Motif Class Proportions', fontweight='bold')

# ============================================================================
# 6. COVERAGE MAP (First sequence)
# ============================================================================
print("6️⃣ Creating Coverage Map...")
ax6 = plt.subplot(2, 3, 6)
if all_results:
    first_seq_motifs = all_results[0]['motifs']
    seq_length = len(all_results[0]['sequence'])
    
    for i, motif in enumerate(first_seq_motifs[:50]):  # Limit to 50 for visibility
        start = motif.get('Start', 0)
        end = motif.get('End', 0)
        ax6.plot([start, end], [i, i], linewidth=2, alpha=0.7)
    
    ax6.set_xlabel('Position (bp)')
    ax6.set_ylabel('Motif Index')
    ax6.set_title(f'Motif Coverage Map\n{all_results[0]["name"]}', fontweight='bold')
    ax6.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('nbdscanner_visualizations.png', dpi=300, bbox_inches='tight')
print("\n✅ Visualization saved: nbdscanner_visualizations.png")
plt.show()

# ============================================================================
# CIRCOS PLOTS
# ============================================================================
print("\n" + "=" * 80)
print("🎯 GENERATING CIRCOS-STYLE PLOTS")
print("=" * 80)

if all_results:
    first_result = all_results[0]
    first_motifs = first_result['motifs']
    first_seq_length = len(first_result['sequence'])
    
    # Radial class density
    print("\n🎯 Creating Radial Class Density Plot...")
    fig_radial = plot_radial_class_density(first_motifs, first_seq_length,
                                           title=f"Radial Density - {first_result['name']}")
    plt.savefig('nbdscanner_radial_density.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig_radial)
    print("✅ Saved: nbdscanner_radial_density.png")
    
    # Stacked density track
    print("\n📊 Creating Stacked Density Track...")
    fig_stacked = plot_stacked_density_track(first_motifs, first_seq_length,
                                             title=f"Stacked Density - {first_result['name']}")
    plt.savefig('nbdscanner_stacked_density.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig_stacked)
    print("✅ Saved: nbdscanner_stacked_density.png")

# ============================================================================
# COMPREHENSIVE SUMMARY TABLE WITH STATISTICS
# ============================================================================
print("\n" + "=" * 80)
print("📋 COMPREHENSIVE ANALYSIS SUMMARY")
print("=" * 80)

summary_data = []
for cls in sorted(class_counts.keys()):
    cls_motifs = [m for m in all_motifs if m.get('Class') == cls]
    if cls_motifs:
        density = len(cls_motifs) / sequence_kb if sequence_kb > 0 else 0
        lengths = [m.get('Length', 0) for m in cls_motifs]
        scores = [m.get('Score', 0) for m in cls_motifs]
        summary_data.append({
            'Class': cls,
            'Count': len(cls_motifs),
            '% of Total': f"{len(cls_motifs) / len(all_motifs) * 100:.1f}%",
            'Density (/kb)': f"{density:.2f}",
            'Avg Length': f"{np.mean(lengths):.1f}",
            'Avg Score': f"{np.mean(scores):.3f}",
            'Len Range': f"{min(lengths)}-{max(lengths)}",
            'Score Range': f"{min(scores):.2f}-{max(scores):.2f}"
        })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

# Final performance summary
print("\n" + "=" * 80)
print("🎉 ANALYSIS COMPLETE - FINAL REPORT")
print("=" * 80)
print("\n⚡ PERFORMANCE METRICS:")
print(f"  ├─ Total Processing Time: {total_time:.2f} seconds")
print(f"  ├─ Processing Speed: {overall_speed:,.0f} bp/second")
print(f"  ├─ Sequences Analyzed: {len(all_results)}")
print(f"  └─ Total Base Pairs: {total_sequence_length:,} bp")
print("\n📊 DETECTION SUMMARY:")
print(f"  ├─ Total Motifs: {len(all_motifs)}")
print(f"  ├─ Unique Classes: {df_results['Class'].nunique()}")
print(f"  ├─ Unique Subclasses: {df_results['Subclass'].nunique()}")
print(f"  └─ Overall Density: {len(all_motifs) / sequence_kb:.2f} motifs/kb")
print("\n📁 FILES GENERATED:")
print("   ├─ nbdscanner_results.xlsx (Excel - multi-sheet)")
print("   ├─ nbdscanner_results.csv (CSV)")
print("   ├─ nbdscanner_results.json (JSON)")
print("   ├─ nbdscanner_visualizations.png (Main Plots)")
print("   ├─ nbdscanner_radial_density.png (Radial Density)")
print("   └─ nbdscanner_stacked_density.png (Stacked Density)")
print("\n✨ All done! Your results are ready for publication.")
print("\n🚀 LOCAL VERSION: No sequence length limits applied.")
print("   (Web version is limited to 1,000,000 nucleotides)")